In [1]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
import shap
from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                              precision_recall_curve)

# Load everything
logreg = joblib.load('../models/logreg.pkl')
rf     = joblib.load('../models/random_forest.pkl')
nn     = tf.keras.models.load_model('../models/neural_network.keras')

X_test_scaled   = pd.read_csv('../data/X_test_scaled.csv')
X_test_unscaled = pd.read_csv('../data/X_test_unscaled.csv')
y_test          = pd.read_csv('../data/y_test.csv').squeeze()

In [2]:
# ── Export 1: Model Metrics Summary ──────────────────────────────────────────
metrics = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': 0.857, 'Precision': 0.571,
     'Recall': 0.426, 'F1': 0.488, 'ROC_AUC': 0.721,
     'Explainability_Score': 5, 'Explainability_Method': 'Native Coefficients',
     'Regulatory_Suitability': 'High', 'Computational_Cost': 'Very Low'},
    {'Model': 'Random Forest', 'Accuracy': 0.806, 'Precision': 0.375,
     'Recall': 0.319, 'F1': 0.345, 'ROC_AUC': 0.731,
     'Explainability_Score': 4, 'Explainability_Method': 'SHAP TreeExplainer',
     'Regulatory_Suitability': 'Medium', 'Computational_Cost': 'Low-Medium'},
    {'Model': 'Neural Network', 'Accuracy': 0.857, 'Precision': 0.571,
     'Recall': 0.426, 'F1': 0.488, 'ROC_AUC': 0.739,
     'Explainability_Score': 2, 'Explainability_Method': 'SHAP KernelExplainer',
     'Regulatory_Suitability': 'Low', 'Computational_Cost': 'High'},
])
metrics.to_csv('../data/powerbi_metrics.csv', index=False)
print("Saved powerbi_metrics.csv")

Saved powerbi_metrics.csv


In [3]:
# ── Export 2: ROC Curve Points ────────────────────────────────────────────────
y_proba_lr = logreg.predict_proba(X_test_scaled)[:, 1]
y_proba_rf = rf.predict_proba(X_test_unscaled)[:, 1]
y_proba_nn = nn.predict(X_test_scaled, verbose=0).flatten()

roc_rows = []
for name, y_prob in [('Logistic Regression', y_proba_lr),
                      ('Random Forest',       y_proba_rf),
                      ('Neural Network',      y_proba_nn)]:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc_val = auc(fpr, tpr)
    # Downsample to 100 points so Power BI chart isn't sluggish
    idx = np.linspace(0, len(fpr)-1, 100, dtype=int)
    for i in idx:
        roc_rows.append({
            'Model': name,
            'FPR':   round(float(fpr[i]), 4),
            'TPR':   round(float(tpr[i]), 4),
            'AUC':   round(roc_auc_val, 3),
            'Model_AUC_Label': f"{name} (AUC={roc_auc_val:.3f})"
        })

pd.DataFrame(roc_rows).to_csv('../data/powerbi_roc_curves.csv', index=False)
print("Saved powerbi_roc_curves.csv")

Saved powerbi_roc_curves.csv


In [4]:
# ── Export 3: Confusion Matrix Data ──────────────────────────────────────────
cm_rows = []
for name, y_prob in [('Logistic Regression', y_proba_lr),
                      ('Random Forest',       y_proba_rf),
                      ('Neural Network',      y_proba_nn)]:
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    labels = [('True Negative', 0, 0), ('False Positive', 0, 1),
              ('False Negative', 1, 0), ('True Positive', 1, 1)]
    for label, row, col in labels:
        cm_rows.append({
            'Model':       name,
            'Category':    label,
            'Actual':      'No Attrition' if row == 0 else 'Attrition',
            'Predicted':   'No Attrition' if col == 0 else 'Attrition',
            'Count':       int(cm[row, col]),
            'Is_Correct':  'Correct' if row == col else 'Incorrect'
        })

pd.DataFrame(cm_rows).to_csv('../data/powerbi_confusion_matrix.csv', index=False)
print("Saved powerbi_confusion_matrix.csv")

Saved powerbi_confusion_matrix.csv


In [5]:
# ── Export 4: Precision-Recall Curve Points ───────────────────────────────────
pr_rows = []
for name, y_prob in [('Logistic Regression', y_proba_lr),
                      ('Random Forest',       y_proba_rf),
                      ('Neural Network',      y_proba_nn)]:
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    idx = np.linspace(0, len(precision)-1, 100, dtype=int)
    for i in idx:
        pr_rows.append({
            'Model':     name,
            'Precision': round(float(precision[i]), 4),
            'Recall':    round(float(recall[i]), 4),
        })

pd.DataFrame(pr_rows).to_csv('../data/powerbi_pr_curves.csv', index=False)
print("Saved powerbi_pr_curves.csv")

Saved powerbi_pr_curves.csv


In [6]:
# ── Export 5: SHAP Feature Importance (all three models) ─────────────────────
shap_rows = []

# Logistic Regression — use absolute coefficient as importance proxy
lr_importances = np.abs(logreg.coef_[0])
for feat, imp in zip(X_test_scaled.columns, lr_importances):
    shap_rows.append({
        'Model':      'Logistic Regression',
        'Feature':    feat,
        'Importance': round(float(imp), 4),
        'Direction':  'Increases Risk' if logreg.coef_[0][list(X_test_scaled.columns).index(feat)] > 0 else 'Decreases Risk'
    })

# Random Forest — SHAP mean absolute values
explainer_rf      = shap.TreeExplainer(rf)
sv_rf_full        = np.array(explainer_rf.shap_values(X_test_unscaled))
sv_rf             = sv_rf_full[:, :, 1]  # class 1
rf_mean_abs_shap  = np.abs(sv_rf).mean(axis=0)
rf_mean_shap      = sv_rf.mean(axis=0)
for feat, imp, direction_val in zip(X_test_unscaled.columns, rf_mean_abs_shap, rf_mean_shap):
    shap_rows.append({
        'Model':      'Random Forest',
        'Feature':    feat,
        'Importance': round(float(imp), 4),
        'Direction':  'Increases Risk' if direction_val > 0 else 'Decreases Risk'
    })

# Neural Network — use coefficients from logreg as proxy for NN importance
# (KernelExplainer on full test set would take too long — use the 50-row sample)
background        = shap.sample(X_test_scaled, 50)
explainer_nn_exp  = shap.KernelExplainer(nn.predict, background)
sv_nn_raw         = np.array(explainer_nn_exp.shap_values(X_test_scaled.iloc[:50]))
if sv_nn_raw.ndim == 3:
    sv_nn = sv_nn_raw[:, :, 0]
else:
    sv_nn = sv_nn_raw
nn_mean_abs_shap  = np.abs(sv_nn).mean(axis=0)
nn_mean_shap      = sv_nn.mean(axis=0)
for feat, imp, direction_val in zip(X_test_scaled.columns, nn_mean_abs_shap, nn_mean_shap):
    shap_rows.append({
        'Model':      'Neural Network',
        'Feature':    feat,
        'Importance': round(float(imp), 4),
        'Direction':  'Increases Risk' if direction_val > 0 else 'Decreases Risk'
    })

shap_df = pd.DataFrame(shap_rows)
shap_df.to_csv('../data/powerbi_shap_importance.csv', index=False)
print("Saved powerbi_shap_importance.csv")
print(shap_df.shape)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


  0%|          | 0/50 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 532us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 552us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 2s 556us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 934us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 912us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 866us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 883us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 934us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 898us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 855us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
3335/3335 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1/1 ━━━━

In [7]:
# ── Export 6: Regulatory Scorecard Data ──────────────────────────────────────
regulatory = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Dimension': 'Transparency',          'Score': 5},
    {'Model': 'Logistic Regression', 'Dimension': 'Auditability',          'Score': 5},
    {'Model': 'Logistic Regression', 'Dimension': 'Explainability',        'Score': 5},
    {'Model': 'Logistic Regression', 'Dimension': 'Human Override Ready',  'Score': 5},
    {'Model': 'Logistic Regression', 'Dimension': 'Maintenance Simplicity','Score': 5},
    {'Model': 'Logistic Regression', 'Dimension': 'Bias Detectability',    'Score': 4},

    {'Model': 'Random Forest',       'Dimension': 'Transparency',          'Score': 3},
    {'Model': 'Random Forest',       'Dimension': 'Auditability',          'Score': 4},
    {'Model': 'Random Forest',       'Dimension': 'Explainability',        'Score': 4},
    {'Model': 'Random Forest',       'Dimension': 'Human Override Ready',  'Score': 4},
    {'Model': 'Random Forest',       'Dimension': 'Maintenance Simplicity','Score': 3},
    {'Model': 'Random Forest',       'Dimension': 'Bias Detectability',    'Score': 3},

    {'Model': 'Neural Network',      'Dimension': 'Transparency',          'Score': 1},
    {'Model': 'Neural Network',      'Dimension': 'Auditability',          'Score': 2},
    {'Model': 'Neural Network',      'Dimension': 'Explainability',        'Score': 2},
    {'Model': 'Neural Network',      'Dimension': 'Human Override Ready',  'Score': 3},
    {'Model': 'Neural Network',      'Dimension': 'Maintenance Simplicity','Score': 1},
    {'Model': 'Neural Network',      'Dimension': 'Bias Detectability',    'Score': 2},
])
regulatory.to_csv('../data/powerbi_regulatory_scorecard.csv', index=False)
print("Saved powerbi_regulatory_scorecard.csv")

Saved powerbi_regulatory_scorecard.csv
